# Map of Science — Figure Workshop

Interactive notebook for exploring and iterating on publication figures.  
Uses the same data loading + parsing from `catalysis_map_figures.py`.

In [ ]:
# ── Setup & Data Loading ────────────────────────────────────────────
import sys, json, os, re
from pathlib import Path
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

# Project style — from examples/notebooks/dev/, repo root is parents[3]
SRC_DIR = str(Path().resolve().parents[3] / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
from llm_synthesis.utils.style_utils import set_style
set_style("manuscript")
plt.rcParams.update({"savefig.dpi": 300, "savefig.bbox": "tight"})

# Import everything from the main script (same folder)
script_dir = str(Path().resolve())
if script_dir not in sys.path:
    sys.path.insert(0, script_dir)

from catalysis_map_figures import (
    load_all_data, llm_parse_all_materials, detect_promoter_pairs,
    get_metal_color, get_support_marker,
    _sorted_metal_legend, _sorted_support_legend,
    classify_synthesis_strategy, interpolate_at_temp,
    STRATEGY_COLORS, STRATEGY_ORDER,
    Y_LABEL, Y_KEYWORDS, REF_TEMP, METRIC_NAME,
)

print("Imports OK")

In [ ]:
# ── Configuration ───────────────────────────────────────────────────
# From examples/notebooks/dev/, results_catalysis is at ../../data/results_catalysis
DATA_DIR = Path().resolve().parents[1] / "data" / "results_catalysis"
OUT_DIR  = Path().resolve().parents[1] / "data" / "figure_visualisation"

USE_LLM = True  # set False to use regex parsing instead

print(f"Data: {DATA_DIR}")
print(f"Output: {OUT_DIR}")

In [ ]:
# ── Load Data ───────────────────────────────────────────────────────
import catalysis_map_figures as cmf
cmf.DATA_DIR = DATA_DIR  # set the module-level DATA_DIR

mat_cache = None
if USE_LLM:
    mat_cache = llm_parse_all_materials(data_dir=DATA_DIR)

df_curves, df_synthesis = load_all_data(material_cache=mat_cache)

# Quick filter: non-plasma, valid metal
df = df_curves[
    (~df_curves["is_plasma"]) &
    (df_curves["metal"].notna()) &
    (df_curves["metal"] != "None") &
    (df_curves["metal"].astype(str) != "nan")
].copy()

print(f"{len(df_curves)} total curves, {len(df)} after filtering")
print(f"{len(df_synthesis)} synthesis records")
print(f"\nMetals:   {sorted(df['metal'].unique())}")
print(f"Supports: {sorted(df['support'].unique())}")

---
## Fig 1 — Full Conversion Landscape
All curves, colored by metal, marker by support.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

metals_present = set()
supports_present = set()

for _, row in df.iterrows():
    coords = np.array(row["coordinates"], dtype=float)
    if len(coords) < 2:
        continue
    temps, convs = coords[:, 0], coords[:, 1]
    metal = row["metal"]
    support = row["support"] if pd.notna(row["support"]) else "Other"
    color = get_metal_color(metal)
    marker = get_support_marker(support)

    ax.plot(temps, convs, color=color, alpha=0.55, linewidth=1.0)
    step = max(1, len(temps) // 5)
    ax.scatter(temps[::step], convs[::step], color=color, marker=marker,
               s=20, zorder=3, edgecolors="white", linewidth=0.3, alpha=0.8)
    metals_present.add(metal)
    supports_present.add(support)

# Legends
metal_order = _sorted_metal_legend(metals_present)
metal_handles = [Line2D([0], [0], color=get_metal_color(m), lw=2, label=m) for m in metal_order]
leg1 = ax.legend(handles=metal_handles, title="Active Metal",
                 loc="lower right", frameon=True, framealpha=0.95,
                 edgecolor="grey", fontsize=7, title_fontsize=8)
ax.add_artist(leg1)

sup_order = _sorted_support_legend(supports_present)
sup_handles = [Line2D([0], [0], marker=get_support_marker(s), color="grey",
                      lw=0, markersize=5, label=s) for s in sup_order]
ax.legend(handles=sup_handles, title="Support",
          loc="center left", frameon=True, framealpha=0.95,
          edgecolor="grey", bbox_to_anchor=(1.02, 0.5), fontsize=7, title_fontsize=8)

ax.set_xlabel("Temperature (°C)")
ax.set_ylabel(Y_LABEL)
ax.set_ylim(-2, 105)
fig.subplots_adjust(right=0.82)
plt.show()

---
## Fig — Conversion Landscape by Metal (with data points, no support markers)
Colored by metal, uniform dot markers at each data point.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

metals_present = set()

for _, row in df.iterrows():
    coords = np.array(row["coordinates"], dtype=float)
    if len(coords) < 2:
        continue
    temps, convs = coords[:, 0], coords[:, 1]
    metal = row["metal"]
    color = get_metal_color(metal)

    ax.plot(temps, convs, color=color, alpha=0.55, linewidth=1.0)
    ax.scatter(temps, convs, color=color, marker="o",
               s=10, zorder=3, edgecolors="white", linewidth=0.2, alpha=0.7)
    metals_present.add(metal)

metal_order = _sorted_metal_legend(metals_present)
metal_handles = [Line2D([0], [0], color=get_metal_color(m), lw=2, label=m) for m in metal_order]
ax.legend(handles=metal_handles, title="Active Metal",
          loc="lower right", frameon=True, framealpha=0.95,
          edgecolor="grey", fontsize=7, title_fontsize=8)

ax.set_xlabel("Temperature (°C)")
ax.set_ylabel(Y_LABEL)
ax.set_ylim(-2, 105)
plt.show()

---
## Fig — Spotlight: Best Performer
All curves in grey, best-performing curve highlighted in its metal color and labeled.

In [ ]:
# Best performer = reaches ≥50% conversion at the lowest temperature (T50)
# Only consider curves whose maximum ever exceeds 50%
best_idx   = None
best_t50   = np.inf

for idx, row in df.iterrows():
    coords = np.array(row["coordinates"], dtype=float)
    if len(coords) < 2:
        continue
    temps, convs = coords[:, 0], coords[:, 1]
    if convs.max() < 50:
        continue
    # Interpolate T50: first temperature where conversion crosses 50%
    # Walk through sorted temps to find the crossing
    for i in range(len(convs) - 1):
        if convs[i] <= 50 <= convs[i + 1]:
            t50 = np.interp(50, [convs[i], convs[i + 1]], [temps[i], temps[i + 1]])
            break
        elif convs[i] >= 50:  # already above 50 from the first point
            t50 = temps[i]
            break
    else:
        continue
    if t50 < best_t50:
        best_t50  = t50
        best_idx  = idx

best_row     = df.loc[best_idx]
best_coords  = np.array(best_row["coordinates"], dtype=float)
best_metal   = best_row["metal"]
best_support = best_row["support"] if pd.notna(best_row["support"]) else "Other"
best_name    = best_row.get("material_name", f"{best_metal}/{best_support}")
best_label   = f"{best_name}\n({best_metal}/{best_support})"
best_color   = get_metal_color(best_metal)
best_temps, best_convs = best_coords[:, 0], best_coords[:, 1]

# Point to annotate: the T50 point on the best curve
ann_t = best_t50
ann_c = 50.0

fig, ax = plt.subplots(figsize=(10, 6))

for _, row in df.iterrows():
    coords = np.array(row["coordinates"], dtype=float)
    if len(coords) < 2:
        continue
    temps, convs = coords[:, 0], coords[:, 1]
    ax.plot(temps, convs, color="#C0C0C0", alpha=0.4, linewidth=0.8, zorder=1)
    ax.scatter(temps, convs, color="#C0C0C0", marker="o",
               s=8, zorder=2, edgecolors="none", alpha=0.4)

# Highlighted best curve
ax.plot(best_temps, best_convs, color=best_color, linewidth=2.2, zorder=4)
ax.scatter(best_temps, best_convs, color=best_color, marker="o",
           s=20, zorder=5, edgecolors="white", linewidth=0.4)

# In-plot annotation with arrow
ax.annotate(
    best_label,
    xy=(ann_t, ann_c),                      # arrow tip: T50 on best curve
    xytext=(ann_t + 40, ann_c + 25),         # text box: up and to the right
    color=best_color,
    fontsize=8,
    fontweight="bold",
    ha="left",
    va="bottom",
    arrowprops=dict(
        arrowstyle="-|>",
        color=best_color,
        lw=1.2,
        connectionstyle="arc3,rad=-0.2",
    ),
    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec=best_color, lw=0.8, alpha=0.9),
)

ax.set_xlabel("Temperature (°C)")
ax.set_ylabel(Y_LABEL)
ax.set_ylim(-2, 105)
plt.show()

---
## Fig — Per-Metal Subplots
One subplot per metal, showing all supports for that metal.

In [ ]:
# Pick which metals to show (sorted by number of curves)
metal_counts = df["metal"].value_counts()
metals_to_plot = metal_counts[metal_counts >= 2].index.tolist()
n = len(metals_to_plot)
ncols = min(4, n)
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.5 * nrows), squeeze=False)

for idx, metal in enumerate(metals_to_plot):
    ax = axes[idx // ncols][idx % ncols]
    sub = df[df["metal"] == metal]
    supports_present = set()

    for _, row in sub.iterrows():
        coords = np.array(row["coordinates"], dtype=float)
        if len(coords) < 2:
            continue
        temps, convs = coords[:, 0], coords[:, 1]
        support = row["support"] if pd.notna(row["support"]) else "Other"
        marker = get_support_marker(support)
        color = get_metal_color(metal)

        ax.plot(temps, convs, color=color, alpha=0.6, linewidth=1.0)
        step = max(1, len(temps) // 4)
        ax.scatter(temps[::step], convs[::step], color=color, marker=marker,
                   s=18, zorder=3, edgecolors="white", linewidth=0.3, alpha=0.8)
        supports_present.add(support)

    ax.set_title(f"{metal} ({len(sub)} curves)", fontsize=9, fontweight="bold")
    ax.set_ylim(-2, 105)
    ax.set_xlabel("T (°C)", fontsize=8)
    ax.set_ylabel(Y_LABEL, fontsize=8)

    if supports_present:
        sup_order = _sorted_support_legend(supports_present)
        sup_handles = [Line2D([0], [0], marker=get_support_marker(s), color=color,
                              lw=0, markersize=4, label=s) for s in sup_order]
        ax.legend(handles=sup_handles, fontsize=5, loc="lower right",
                  frameon=True, framealpha=0.9, edgecolor="grey")

# Hide empty subplots
for idx in range(n, nrows * ncols):
    axes[idx // ncols][idx % ncols].set_visible(False)

fig.tight_layout()
plt.show()

---
## Fig — Per-Metal Subplots, Colored by Support
Same layout as above, but curves are colored by support material instead of metal.

In [ ]:
# Build a consistent color palette for all supports across the dataset
all_supports = sorted(df["support"].dropna().unique())
support_cmap = plt.cm.get_cmap("tab20", len(all_supports))
SUPPORT_COLORS = {s: support_cmap(i) for i, s in enumerate(all_supports)}

metal_counts = df["metal"].value_counts()
metals_to_plot = metal_counts[metal_counts >= 2].index.tolist()
n = len(metals_to_plot)
ncols = min(4, n)
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.5 * nrows), squeeze=False)

for idx, metal in enumerate(metals_to_plot):
    ax = axes[idx // ncols][idx % ncols]
    sub = df[df["metal"] == metal]
    supports_present = set()

    for _, row in sub.iterrows():
        coords = np.array(row["coordinates"], dtype=float)
        if len(coords) < 2:
            continue
        temps, convs = coords[:, 0], coords[:, 1]
        support = row["support"] if pd.notna(row["support"]) else "Other"
        color = SUPPORT_COLORS.get(support, "grey")

        ax.plot(temps, convs, color=color, alpha=0.6, linewidth=1.0)
        ax.scatter(temps, convs, color=color, marker="o",
                   s=10, zorder=3, edgecolors="white", linewidth=0.2, alpha=0.7)
        supports_present.add(support)

    ax.set_title(f"{metal} ({len(sub)} curves)", fontsize=9, fontweight="bold")
    ax.set_ylim(-2, 105)
    ax.set_xlabel("T (°C)", fontsize=8)
    ax.set_ylabel(Y_LABEL, fontsize=8)

    if supports_present:
        sup_order = _sorted_support_legend(supports_present)
        sup_handles = [Line2D([0], [0], color=SUPPORT_COLORS.get(s, "grey"),
                              lw=2, label=s) for s in sup_order]
        ax.legend(handles=sup_handles, fontsize=5, loc="lower right",
                  frameon=True, framealpha=0.9, edgecolor="grey")

for idx in range(n, nrows * ncols):
    axes[idx // ncols][idx % ncols].set_visible(False)

fig.tight_layout()
plt.show()

---
## Fig — Per-Metal Group Subplots
Group metals into families (noble, base, bimetallic) for a cleaner overview.

In [ ]:
# Define metal groups — customise as needed
METAL_GROUPS = {
    "Ru-based":    ["Ru", "RuFe", "RuNi", "NiRu"],
    "Ni-based":    ["Ni", "NiCo", "NiRu"],
    "Fe-based":    ["Fe", "FeCo", "FeNi"],
    "Co-based":    ["Co", "CoMo", "CoNi"],
}

fig, axes = plt.subplots(1, len(METAL_GROUPS), figsize=(4 * len(METAL_GROUPS), 4), squeeze=False)

for idx, (group_name, metals) in enumerate(METAL_GROUPS.items()):
    ax = axes[0][idx]
    sub = df[df["metal"].isin(metals)]
    metals_in_group = set()

    for _, row in sub.iterrows():
        coords = np.array(row["coordinates"], dtype=float)
        if len(coords) < 2:
            continue
        temps, convs = coords[:, 0], coords[:, 1]
        metal = row["metal"]
        color = get_metal_color(metal)
        ax.plot(temps, convs, color=color, alpha=0.6, linewidth=1.0)
        metals_in_group.add(metal)

    ax.set_title(f"{group_name} ({len(sub)} curves)", fontsize=9, fontweight="bold")
    ax.set_ylim(-2, 105)
    ax.set_xlabel("T (°C)", fontsize=8)
    if idx == 0:
        ax.set_ylabel(Y_LABEL, fontsize=8)

    if metals_in_group:
        handles = [Line2D([0], [0], color=get_metal_color(m), lw=2, label=m)
                   for m in sorted(metals_in_group)]
        ax.legend(handles=handles, fontsize=6, loc="lower right",
                  frameon=True, framealpha=0.9, edgecolor="grey")

fig.tight_layout()
plt.show()

---
## Fig — Conversion Landscape by Synthesis Strategy

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

strategy_counts = {}
for strat in STRATEGY_ORDER:
    sub = df[df["strategy"] == strat]
    strategy_counts[strat] = len(sub)
    color = STRATEGY_COLORS[strat]
    for _, row in sub.iterrows():
        coords = np.array(row["coordinates"], dtype=float)
        if len(coords) < 2:
            continue
        temps, convs = coords[:, 0], coords[:, 1]
        ax.plot(temps, convs, color=color, alpha=0.5, linewidth=1.0)

handles = [Line2D([0], [0], color=STRATEGY_COLORS[s], lw=2,
                  label=f"{s} ({strategy_counts.get(s, 0)})")
           for s in STRATEGY_ORDER if strategy_counts.get(s, 0) > 0]
ax.legend(handles=handles, title="Synthesis Strategy",
          loc="lower right", frameon=True, fontsize=8, title_fontsize=9)
ax.set_xlabel("Temperature (°C)")
ax.set_ylabel(Y_LABEL)
ax.set_ylim(-2, 105)
plt.show()

---
## Fig — Per-Support Subplots
One subplot per support, showing all metals on that support.

In [ ]:
support_counts = df["support"].value_counts()
supports_to_plot = support_counts[support_counts >= 3].index.tolist()
n = len(supports_to_plot)
ncols = min(4, n)
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.5 * nrows), squeeze=False)

for idx, support in enumerate(supports_to_plot):
    ax = axes[idx // ncols][idx % ncols]
    sub = df[df["support"] == support]
    metals_present = set()

    for _, row in sub.iterrows():
        coords = np.array(row["coordinates"], dtype=float)
        if len(coords) < 2:
            continue
        temps, convs = coords[:, 0], coords[:, 1]
        metal = row["metal"]
        color = get_metal_color(metal)
        ax.plot(temps, convs, color=color, alpha=0.6, linewidth=1.0)
        metals_present.add(metal)

    ax.set_title(f"{support} ({len(sub)} curves)", fontsize=9, fontweight="bold")
    ax.set_ylim(-2, 105)
    ax.set_xlabel("T (°C)", fontsize=8)
    ax.set_ylabel(Y_LABEL, fontsize=8)

    if metals_present:
        handles = [Line2D([0], [0], color=get_metal_color(m), lw=2, label=m)
                   for m in sorted(metals_present)]
        ax.legend(handles=handles, fontsize=5, loc="lower right",
                  frameon=True, framealpha=0.9, edgecolor="grey")

for idx in range(n, nrows * ncols):
    axes[idx // ncols][idx % ncols].set_visible(False)

fig.tight_layout()
plt.show()

---
## Fig — Bar Chart: Best Performance per Metal at Reference Temperature

In [ ]:
best_per_metal = (
    df.dropna(subset=["conv_at_500"])
    .groupby("metal")["conv_at_500"]
    .max()
    .sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(6, max(3, len(best_per_metal) * 0.35)))
colors = [get_metal_color(m) for m in best_per_metal.index]
ax.barh(best_per_metal.index, best_per_metal.values, color=colors, edgecolor="white")
ax.set_xlabel(f"Best {METRIC_NAME} at {REF_TEMP:.0f} °C (%)")
ax.set_xlim(0, 105)
for i, (metal, val) in enumerate(best_per_metal.items()):
    ax.text(val + 1, i, f"{val:.0f}%", va="center", fontsize=7)
fig.tight_layout()
plt.show()

---
## Fig — Box Plot: Performance Distribution per Metal

In [ ]:
conv_data = df.dropna(subset=["conv_at_500"])
metal_order = conv_data.groupby("metal")["conv_at_500"].median().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(10, 5))
box_data = [conv_data[conv_data["metal"] == m]["conv_at_500"].values for m in metal_order]
bp = ax.boxplot(box_data, labels=metal_order, patch_artist=True, vert=True)

for patch, metal in zip(bp["boxes"], metal_order):
    patch.set_facecolor(get_metal_color(metal))
    patch.set_alpha(0.7)

ax.set_ylabel(f"{METRIC_NAME.capitalize()} at {REF_TEMP:.0f} °C (%)")
ax.set_xlabel("Metal")
plt.xticks(rotation=45, ha="right")
fig.tight_layout()
plt.show()

---
## Scratch Cell
Use this for quick experiments.

---
## Fig — Heatmaps: Conversion Landscape
Two views: Metal × Support (at reference temp) and Metal × Temperature (binned median).

In [ ]:
from llm_synthesis.utils.style_utils import get_cmap
from matplotlib.colors import LinearSegmentedColormap

# Build a sequential colormap from the style palette (light → dark)
_pal = get_cmap().colors  # list of RGBA from the ListedColormap
# Pick a light and a dark anchor from the palette for a clean sequential ramp
_seq_cmap = LinearSegmentedColormap.from_list(
    "style_seq", [_pal[5], _pal[2], _pal[12]], N=256  # light peach → blue → purple
)

# ── Heatmap 1: Metal × Support — median conversion at REF_TEMP ──────
pivot_ms = (
    df.dropna(subset=["conv_at_500", "support"])
    .groupby(["metal", "support"])["conv_at_500"]
    .median()
    .unstack()
)
# Sort rows/cols by median for readability
pivot_ms = pivot_ms.loc[
    pivot_ms.mean(axis=1).sort_values(ascending=False).index,
    pivot_ms.mean(axis=0).sort_values(ascending=False).index,
]

fig, ax = plt.subplots(figsize=(max(7, len(pivot_ms.columns) * 0.55),
                                max(4, len(pivot_ms) * 0.5)))
im = ax.imshow(pivot_ms.values, aspect="auto", cmap=_seq_cmap, vmin=0, vmax=100)

ax.set_xticks(range(len(pivot_ms.columns)))
ax.set_xticklabels(pivot_ms.columns, rotation=45, ha="right", fontsize=7)
ax.set_yticks(range(len(pivot_ms.index)))
ax.set_yticklabels(pivot_ms.index, fontsize=8)

for i in range(len(pivot_ms.index)):
    for j in range(len(pivot_ms.columns)):
        val = pivot_ms.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f"{val:.0f}", ha="center", va="center",
                    fontsize=6, color="white" if val > 55 else "black")

plt.colorbar(im, ax=ax, label=f"Median {METRIC_NAME} at {REF_TEMP:.0f} °C (%)", shrink=0.8)
ax.set_title("Conversion Landscape: Metal × Support", fontsize=10)
fig.tight_layout()
plt.show()

# ── Heatmap 2: Metal × Temperature — binned median conversion ────────
T_BINS = np.arange(200, 651, 50)
bin_centers = (T_BINS[:-1] + T_BINS[1:]) / 2

records = []
for _, row in df.iterrows():
    coords = np.array(row["coordinates"], dtype=float)
    if len(coords) < 2:
        continue
    for tc in bin_centers:
        val = interpolate_at_temp(coords, tc)
        if not np.isnan(val):
            records.append({"metal": row["metal"], "T_bin": tc, "conv": val})

df_bins = pd.DataFrame(records)
pivot_mt = df_bins.groupby(["metal", "T_bin"])["conv"].median().unstack()
# Sort rows by overall median conversion
pivot_mt = pivot_mt.loc[pivot_mt.mean(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(max(7, len(pivot_mt.columns) * 0.7),
                                max(4, len(pivot_mt) * 0.5)))
im = ax.imshow(pivot_mt.values, aspect="auto", cmap=_seq_cmap, vmin=0, vmax=100)

ax.set_xticks(range(len(pivot_mt.columns)))
ax.set_xticklabels([f"{int(t)}" for t in pivot_mt.columns], fontsize=8)
ax.set_yticks(range(len(pivot_mt.index)))
ax.set_yticklabels(pivot_mt.index, fontsize=8)

for i in range(len(pivot_mt.index)):
    for j in range(len(pivot_mt.columns)):
        val = pivot_mt.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f"{val:.0f}", ha="center", va="center",
                    fontsize=6, color="white" if val > 55 else "black")

plt.colorbar(im, ax=ax, label=f"Median {METRIC_NAME} (%)", shrink=0.8)
ax.set_xlabel("Temperature (°C)", fontsize=9)
ax.set_title("Conversion Landscape: Metal × Temperature", fontsize=10)
fig.tight_layout()
plt.show()

In [ ]:
# Quick look at the data
df[["paper_dir", "material_name", "metal", "support", "conv_at_500", "strategy"]].head(20)

In [ ]:
# Save a figure — just uncomment and run after the cell you want to save
# fig.savefig(OUT_DIR / "my_figure.png")
# fig.savefig(OUT_DIR / "my_figure.pdf")
# print("Saved!")